# Solving the Steady-State Radiative Transfer Equation (RTE/RFE) using PINNs

In this notebook, we solve the steady-state (time-independent) Radiative Transfer/Flow Equation (RTE/RFE) using a Physics-Informed Neural Network (PINN).


### Case 1 (1D) 
We first focus on a simple case , which is a 1D space + 1D direction test case with the following configuration :
* **Geometry:** $0 \le x \le L_x$ with $L_x = 1.0\text{ m}$.
* **Equation (1D RTE):**
  $$\mu \frac{\partial I}{\partial x} + (\kappa + \sigma(x)) I(x, \mu) = \frac{\sigma(x)}{2} \int_{-1}^{1} I(x, \mu') d\mu'$$
* **Physical Properties:**
  * Absorption coefficient: $\kappa = 0$ (Non-absorbing medium).
  * Scattering coefficient: $\sigma(x) = \frac{1}{x}$.
* **Boundary Conditions (BC):**
  * At $x = 0$ (for incoming directions $\mu > 0$): $I(0, \mu) = 1$
  * At $x = L_x = 1$ (for incoming directions $\mu < 0$): $I(1, \mu) = 0$

### Vanilla PINN Approach (Soft Constraints)
Unlike previous chapters where we enforced boundary conditions exactly by construction (hard constraints: $u = A + B \cdot N$), doing so for the RTE is highly complex. The boundary conditions depend directly on the propagation direction (incoming vs. outgoing directions), making it difficult to construct analytical boundary functions $A$ and $B$.

Consequently, we begin with a **Vanilla PINN** approach .



In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

## 1. PINN Architecture Definition
Definition of the standard Neural Network (Multi-Layer Perceptron) to approximate the solution $u(t,x)$.

In [2]:
class PinnRFEEq(nn.Module):
    def __init__(self):
        super().__init__()
        self.couche_entree = nn.Linear(2, 50)
        self.couche_cachee1 = nn.Linear(50, 50)
        self.couche_cachee2 = nn.Linear(50, 50)
        self.couche_sortie = nn.Linear(50, 1)

    def forward(self, x):
        x = torch.tanh(self.couche_entree(x))
        x = torch.tanh(self.couche_cachee1(x))
        x = torch.tanh(self.couche_cachee2(x))
        return self.couche_sortie(x)

## 2. Sampling and Collocation Points Generation
Functions to generate collocation points (in the domain) ,and boundary points ($x=0$ and $x=1$).

In [3]:
mu_min, mu_max = -1.0, 1.0
x_min, x_max = 0.0, 1.0


def generer_points_collocation(n_pde):
    mu_colloc = torch.rand(n_pde, 1) * (mu_max - mu_min) + mu_min
    x_colloc = torch.rand(n_pde, 1) * (x_max - x_min) + x_min
    return mu_colloc.float(), x_colloc.float()



def generer_points_bords(n_bords):                                                                       
    n_half = n_bords // 2                                                                                
                                                                                                             
    x_gauche = torch.zeros(n_half, 1)                                                                    
    mu_gauche = torch.rand(n_half, 1)                                                 
                                                                                                             
    x_droite = torch.ones(n_half, 1)                                                                     
    mu_droite = -torch.rand(n_half, 1)                                            
                                                                                                             
    return x_gauche.float(), mu_gauche.float(), x_droite.float(), mu_droite.float()   

## 3. Loss Functions Definition
Grouping of loss calculation functions for initial conditions (IV), boundary conditions (BC), and the PDE residual (CLP).

In [14]:
def calc_bc_loss(model, x_g, mu_g, x_d, mu_d):                                                           
                                                                                                             
    inputs_gauche = torch.cat([x_g, mu_g], dim=1)                                            
    inputs_droite = torch.cat([x_d, mu_d], dim=1)                                             
                                                                                     
    pred_gauche = model(inputs_gauche)                                                                   
    pred_droite = model(inputs_droite)                                                                   
                                                                                                             
    loss_bc_gauche = torch.mean((pred_gauche - 1.0) ** 2)                                                
    loss_bc_droite = torch.mean((pred_droite - 0.0) ** 2)                                                
                                                                                                             
    return loss_bc_gauche + loss_bc_droite           

def calc_clp_loss(model, mu_colloc, x_colloc):
    mu_colloc.requires_grad_(True)
    x_colloc.requires_grad_(True)

    I_pred = model(torch.cat([x_colloc, mu_colloc], dim=1))

    
    I_x = torch.autograd.grad(
        outputs=I_pred,
        inputs=x_colloc,
        grad_outputs=torch.ones_like(I_pred),
        create_graph=True,
    )[0]

    N = x_colloc.shape[0]                                                                                    
                                                                                                             
    x_expanded = x_colloc.repeat(1, N_quadrature)                                                                     
                                                                                                             
    mu_expanded = quad_mu.t().repeat(N, 1)                                                                   
                                                                                                             
                                                                                   
    inputs = torch.cat([x_expanded.flatten().view(-1, 1), mu_expanded.flatten().view(-1, 1)], dim=1)         
    I_quad_preds = model(inputs).view(N, N_quadrature)                                               
                                                                                                                                                                                  
    integral = torch.sum(I_quad_preds * quad_w.t(), dim=1, keepdim=True)
    sigma_x = 1.0 / x_colloc                                                                

    pde_residual = mu_colloc * I_x + sigma_x * I_pred - 0.5 * sigma_x * integral
    loss_pde = torch.mean(pde_residual ** 2) 

    return loss_pde

## 4. Hardware (Device), Model, and Optimizer Initialization
Hardware detection, model creation, and optimizer definition.

In [15]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(device)

modele = PinnRFEEq().to(device)
optimizer = optim.Adam(modele.parameters(), lr=0.001, weight_decay=1e-5)

mu_colloc, x_colloc = generer_points_collocation(12000)


mu_colloc, x_colloc = mu_colloc.to(device), x_colloc.to(device)
n_bords = 1000
x_g, mu_g, x_d, mu_d = generer_points_bords(n_bords) 
x_g = x_g.to(device)                                                                                     
mu_g = mu_g.to(device)                                                                                   
x_d = x_d.to(device)                                                                                     
mu_d = mu_d.to(device)                                                                                   
                                                                                                             


mps


In [16]:
N_quadrature = 16
nodes, weights = np.polynomial.legendre.leggauss(N_quadrature)                                                    
                                                                                                             
quad_mu = torch.tensor(nodes, dtype=torch.float32).view(N_quadrature, 1).to(device)                     
quad_w = torch.tensor(weights, dtype=torch.float32).view(N_quadrature, 1).to(device)


## 5. Model Training
Training phase of the PINN model using the defined optimizers (Adam and/or L-BFGS).

In [17]:
epochs = 500

for epoch in range(epochs):
    optimizer.zero_grad()
    loss_bc = calc_bc_loss(modele, x_g, mu_g, x_d, mu_d)   
    loss_clp = calc_clp_loss(modele, mu_colloc, x_colloc)
    loss_totale = loss_bc + loss_clp
    loss_totale.backward()
    optimizer.step()

    if epoch % 100 == 0:
        print(f"Epoque {epoch:05d} | "
              f"Loss totale: {loss_totale.item():.2e} | "
              f"CLP: {loss_clp.item():.2e} | "
              f"BC: {loss_bc.item():.2e} | "
              )

Epoque 00000 | Loss totale: 6.67e+00 | CLP: 5.75e+00 | BC: 9.21e-01 | 
Epoque 00100 | Loss totale: 3.37e-01 | CLP: 1.12e-02 | BC: 3.26e-01 | 
Epoque 00200 | Loss totale: 1.68e-01 | CLP: 8.58e-02 | BC: 8.18e-02 | 
Epoque 00300 | Loss totale: 1.45e-01 | CLP: 9.57e-02 | BC: 4.88e-02 | 
Epoque 00400 | Loss totale: 1.27e-01 | CLP: 8.63e-02 | BC: 4.11e-02 | 


## 6. Visualizing the Results
Comparison of the solution learned by the PINN with the exact analytical solution of the heat equation.

In [18]:
import sys; sys.path.append("..")
import numpy as np                                                                                                    
from pinnplot import plot_solution                        

u_exact = lambda t, x: np.sin(2*np.pi*x) * np.exp(-(2*np.pi)**2 * alpha * t)

plot_solution(modele, u_exact)           


NameError: name 'alpha' is not defined